In [1]:
# ============================================================
# CELL 1 (FIXED): Load mT5 + reproduce test split + sanity check
# ============================================================
import os, json
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

os.chdir('/kaggle/working')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Device:", device)

MODEL_DIR = "/kaggle/input/datasets/siddhanttimalsina/mt5-model"
DATA_DIR  = "/kaggle/input/datasets/siddhanttimalsina/mt5-dataset/"

# ---------- Load ----------
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_DIR).to(device).eval()
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
print(f"mT5 loaded        : {model.num_parameters()/1e6:.1f}M params (mt5-small arch, untied emb)")
print(f"Tokenizer         : {type(tokenizer).__name__}, vocab={tokenizer.vocab_size}")
print(f"decoder_start=pad = {model.config.decoder_start_token_id} "
      f"({'✓ T5 standard' if model.config.decoder_start_token_id == model.config.pad_token_id else '✗ mismatch'})")

# ---------- Reproduce EXACT test split from mt5_train.py ----------
small  = pd.read_csv(DATA_DIR + "small.csv")
medium = pd.read_csv(DATA_DIR + "medium.csv")
large  = pd.read_csv(DATA_DIR + "large.csv")
all_data = pd.concat([small, medium, large], ignore_index=True)
print(f"\nall_data rows (raw): {len(all_data)}")

_, temp_df = train_test_split(all_data, test_size=0.2, random_state=42)
_, test_df = train_test_split(temp_df, test_size=0.5, random_state=42)
print(f"mT5 test split: {len(test_df)} rows")

before = len(test_df)
test_df = test_df.dropna(subset=['content', 'summary']).copy()
test_df['content'] = test_df['content'].astype(str).str.strip()
test_df['summary'] = test_df['summary'].astype(str).str.strip()
test_df = test_df[(test_df['content'].str.len() > 0) &
                  (test_df['summary'].str.len() > 0)].reset_index(drop=True)
print(f"After NaN/empty filter: {len(test_df)} rows (dropped {before - len(test_df)})")

test_df[['content', 'summary']].to_csv(
    '/kaggle/working/mt5_test_raw.csv', index=False, encoding='utf-8')
print("Saved mt5_test_raw.csv")

# ---------- Sanity generation ----------
sample = test_df.iloc[0]
inputs = tokenizer(
    sample['content'],
    return_tensors='pt', max_length=512, truncation=True
).to(device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_length=128, num_beams=4, early_stopping=True,
    )
pred = tokenizer.decode(outputs[0], skip_special_tokens=True)

print("\n=== Sanity check ===")
print(f"REF  ({len(sample['summary'])} chars): {sample['summary'][:300]}")
print(f"PRED ({len(pred)} chars): {pred[:300]}")

Device: cuda


Loading weights:   0%|          | 0/192 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to encoder.embed_tokens.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.
[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to decoder.embed_tokens.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.
[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


KeyboardInterrupt: 

In [ ]:
# ============================================================
# CELL 2: Generate predictions on all 5,053 mT5 test rows
# ============================================================
import os, time
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from tqdm.auto import tqdm

os.chdir('/kaggle/working')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Reload (idempotent — safe if session restarts between cells)
MODEL_DIR = "/kaggle/input/datasets/siddhanttimalsina/mt5-model"
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_DIR).to(device).eval()
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)

test_df = pd.read_csv('/kaggle/working/mt5_test_raw.csv')
print(f"Test rows: {len(test_df)}")

# ---------- Dataset ----------
MAX_INPUT_LEN = 512

class SummarizationTestSet(Dataset):
    def __init__(self, frame): self.texts = frame['content'].astype(str).tolist()
    def __len__(self): return len(self.texts)
    def __getitem__(self, i): return self.texts[i]

def collate_fn(batch):
    return tokenizer(
        batch,
        max_length=MAX_INPUT_LEN,
        truncation=True,
        padding=True,
        return_tensors='pt',
    )

test_loader = DataLoader(
    SummarizationTestSet(test_df), batch_size=8, shuffle=False,
    collate_fn=collate_fn, num_workers=2, pin_memory=True,
)

# ---------- Decoding config (baseline + repetition guard) ----------
GEN_KWARGS = dict(
    max_length=128,
    num_beams=4,
    no_repeat_ngram_size=3,
    early_stopping=True,
    pad_token_id=tokenizer.pad_token_id,
    eos_token_id=tokenizer.eos_token_id,
    decoder_start_token_id=tokenizer.pad_token_id,
)

# ---------- Smoke test ----------
print("Smoke test on first batch (8 samples)...")
first = next(iter(test_loader))
with torch.no_grad():
    out = model.generate(first['input_ids'].to(device),
                         attention_mask=first['attention_mask'].to(device),
                         **GEN_KWARGS)
prev = tokenizer.batch_decode(out, skip_special_tokens=True)
print(f"  shape: {tuple(out.shape)}")
print(f"  len[0]={len(prev[0])} chars: {prev[0][:150]}")
print("Smoke passed.\n")

# ---------- Full generation ----------
all_preds = []
t0 = time.time()

for batch in tqdm(test_loader, desc="Generating mT5"):
    input_ids      = batch['input_ids'].to(device)
    attention_mask = batch['attention_mask'].to(device)

    with torch.no_grad():
        gen_ids = model.generate(
            input_ids,
            attention_mask=attention_mask,
            **GEN_KWARGS,
        )
    all_preds.extend(tokenizer.batch_decode(gen_ids, skip_special_tokens=True))

elapsed = time.time() - t0
print(f"\nGeneration done in {elapsed/60:.1f} min")

# ---------- Save ----------
pred_df = pd.DataFrame({
    'source':    test_df['content'].astype(str).tolist(),
    'reference': test_df['summary'].astype(str).tolist(),
    'predicted': all_preds,
})
pred_df.to_csv('/kaggle/working/test_predictions_mt5.csv',
               index=False, encoding='utf-8')

empty   = (pred_df['predicted'].str.strip() == '').sum()
avg_len = pred_df['predicted'].str.len().mean()
ref_len = pred_df['reference'].str.len().mean()
print(f"\nSaved test_predictions_mt5.csv ({len(pred_df)} rows)")
print(f"  Empty predictions : {empty} / {len(pred_df)}")
print(f"  Avg PRED length   : {avg_len:.1f} chars")
print(f"  Avg REF  length   : {ref_len:.1f} chars")
print(f"  PRED/REF ratio    : {avg_len/ref_len:.2f}")

print("\nFirst 2 samples:")
for i in range(2):
    print(f"\n--- sample {i+1} ---")
    print(f"REF : {pred_df['reference'].iloc[i][:200]}")
    print(f"PRED: {pred_df['predicted'].iloc[i][:200]}")

In [ ]:
# ============================================================
# CELL 3: Compute 5-metric scores for mT5
# ============================================================
!pip install -q sacrebleu bert-score

import pandas as pd, numpy as np
from collections import Counter
from sacrebleu.metrics import BLEU
from bert_score import score as bert_score_fn
import torch

device = 'cuda' if torch.cuda.is_available() else 'cpu'
pred_df = pd.read_csv('/kaggle/working/test_predictions_mt5.csv')

# Handle any empties (mT5 had 0, defensive)
n_empty = int((pred_df['predicted'].astype(str).str.strip() == '').sum())
if n_empty > 0:
    pred_df.loc[pred_df['predicted'].astype(str).str.strip() == '', 'predicted'] = '।'

refs = pred_df['reference'].astype(str).tolist()
hyps = pred_df['predicted'].astype(str).tolist()
print(f"Evaluating {len(refs)} predictions (empty: {n_empty})\n")

# ---------- 1. BLEU-4 (flores200) — same tokenizer as mBERT ----------
bleu_flores = BLEU(tokenize='flores200', max_ngram_order=4).corpus_score(hyps, [refs]).score
bleu_13a    = BLEU(tokenize='13a',      max_ngram_order=4).corpus_score(hyps, [refs]).score

# ---------- 2–4. ROUGE (manual) ----------
def ng(t, n): return [tuple(t[i:i+n]) for i in range(len(t)-n+1)]
def rn(pt, rt, n):
    p, r = ng(pt, n), ng(rt, n)
    if not p or not r: return 0.0
    ov = sum((Counter(p) & Counter(r)).values())
    pr, rc = ov/len(p), ov/len(r)
    return 0.0 if pr+rc == 0 else 2*pr*rc/(pr+rc)
def rl(pt, rt):
    m, n = len(pt), len(rt)
    if m == 0 or n == 0: return 0.0
    dp = [[0]*(n+1) for _ in range(m+1)]
    for i in range(1, m+1):
        for j in range(1, n+1):
            dp[i][j] = dp[i-1][j-1]+1 if pt[i-1]==rt[j-1] else max(dp[i-1][j], dp[i][j-1])
    lcs = dp[m][n]; pr, rc = lcs/m, lcs/n
    return 0.0 if pr+rc == 0 else 2*pr*rc/(pr+rc)

r1 = float(np.mean([rn(h.split(), r.split(), 1) for h, r in zip(hyps, refs)]))
r2 = float(np.mean([rn(h.split(), r.split(), 2) for h, r in zip(hyps, refs)]))
rL = float(np.mean([rl(h.split(), r.split())       for h, r in zip(hyps, refs)]))

# ---------- 5. BERTScore ----------
P, R, F1 = bert_score_fn(hyps, refs, lang='ne', verbose=False, device=device)
bs_p, bs_r, bs_f1 = P.mean().item(), R.mean().item(), F1.mean().item()

# ---------- Jaccard ----------
def set_jac(A, B):
    if not A and not B: return 1.0
    if not A or not B:  return 0.0
    return len(A & B) / len(A | B)

def jac_word(a, b): return set_jac(set(str(a).split()), set(str(b).split()))
def jac_char(a, b):
    a, b = str(a), str(b)
    return set_jac(set(a[i:i+2] for i in range(max(len(a)-1, 0))),
                   set(b[i:i+2] for i in range(max(len(b)-1, 0))))

jac_w = float(np.mean([jac_word(h, r) for h, r in zip(hyps, refs)]))
jac_c = float(np.mean([jac_char(h, r) for h, r in zip(hyps, refs)]))

# ---------- Print ----------
print("=" * 60)
print("  mT5-small (fine-tuned) — NEPALI SUMMARIZATION")
print("=" * 60)
print(f"  SacreBLEU (BLEU-4, flores200)   : {bleu_flores:.4f}")
print(f"  SacreBLEU (BLEU-4, 13a)         : {bleu_13a:.4f}")
print(f"  ROUGE-1                         : {r1:.4f}")
print(f"  ROUGE-2                         : {r2:.4f}")
print(f"  ROUGE-L                         : {rL:.4f}")
print(f"  BERTScore Precision             : {bs_p:.4f}")
print(f"  BERTScore Recall                : {bs_r:.4f}")
print(f"  BERTScore F1                    : {bs_f1:.4f}")
print(f"  Jaccard (word-unigram)          : {jac_w:.4f}")
print(f"  Jaccard (char-bigram)           : {jac_c:.4f}")
print("=" * 60)
print(f"  Test samples                    : {len(refs)}")
print(f"  Empty predictions               : {n_empty}")
print(f"  Avg PRED length                 : {np.mean([len(h) for h in hyps]):.0f} chars")
print(f"  Avg REF  length                 : {np.mean([len(r) for r in refs]):.0f} chars")
print("=" * 60)

# ---------- Save ----------
pd.Series({
    'model': 'mT5-small-finetuned',
    'sacrebleu_flores200': round(bleu_flores, 4),
    'sacrebleu_13a':       round(bleu_13a, 4),
    'rouge1':              round(r1, 4),
    'rouge2':              round(r2, 4),
    'rougeL':              round(rL, 4),
    'bertscore_p':         round(bs_p, 4),
    'bertscore_r':         round(bs_r, 4),
    'bertscore_f1':        round(bs_f1, 4),
    'jaccard_word':        round(jac_w, 4),
    'jaccard_char':        round(jac_c, 4),
    'n_samples':           len(refs),
    'n_empty':             n_empty,
}).to_csv('/kaggle/working/final_scores_mt5.csv')
print("\nSaved final_scores_mt5.csv")